<a href="https://colab.research.google.com/github/ethnicgarbage/ds2002-fa26/blob/main/notebooks/02-sql-databases/Jun_Seo_Lee_2026_09_11_%E2%80%94_SQL_Challenge_Set_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [44]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [45]:
q1 = q('''
select t.title, a.name, a.country
from tracks t
join artists a on a.artist_id = t.artist_id
''')

q1

,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [46]:
q2 = q('''
select t.genre, avg(t.seconds) as avg_length
from tracks t
group by t.genre
''')

q2

,genre,avg_length
0,None,150.0
1,Electronic,287.5
2,Folk,203.0
3,Latin,210.0
4,Pop,220.5


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [53]:
q3 = q('''
select p.user, count(p.play_id) as plays, count(distinct p.track_id) as number_of_distinct_tracks
from plays p
group by p.user
''')

q3

,user,plays,number_of_distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [48]:
q4 = q('''
select t.title, p.play_id
from tracks t
left join plays p on p.track_id = t.track_id
where p.track_id is null
''')

q4

,title,play_id
0,Ridgeline,None
1,Untitled Demo,None


### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [49]:
q5 = q('''
select a.name, round(sum(t.seconds)/60, 1) as listening_time_minutes
from artists a
join tracks t on t.artist_id = a.artist_id
join plays p on p.track_id = t.track_id
group by a.name
order by listening_time_minutes desc
''')

q5

,name,listening_time_minutes
0,Kestrel,19.0
1,Nova Waves,14.0
2,The Blue Ridge,6.0
3,Marisol,3.0


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [50]:
q6 = q('''
select t.track_id, t.title, t.genre
from tracks t
''')

q6

## With "WHERE genre !=  'Pop', the songs that are not in the pop genre such as Electronic, Latin, Folk, and None will be the only ones that populate. The pop songs such as Skyline and Undertow will not populate.

,track_id,title,genre
0,10,Skyline,Pop
1,11,Undertow,Pop
2,12,Foothills,Folk
3,13,Aurora,Electronic
4,14,Nightfall,Electronic
5,15,Sol,Latin
6,16,Coastline,Folk
7,17,Ridgeline,Folk
8,18,Untitled Demo,None


### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [51]:
q7 = q('''
select p.played_on as date, count(p.play_id) as number_of_plays, count(distinct p.user) as number_of_distinct_users
from plays p
group by p.played_on
order by p.played_on asc
''')

q7

,date,number_of_plays,number_of_distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [54]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

The question that was the most difficult for me was Q4 with the left join. I had trouble figuring out how to effectively combine the two tables. Especially as I wanted to output the plays not as null but as count(p.play_id) but this resulted in only one of the tracks being outputed as 0 and the other (the untitled demo track) not populating at all.

I wasn't sure how to fix this so I reverted back to just p.play_id.